# AutoScope: parasite box detectors on ColabTrains **YOLO26** and **Faster R-CNN** to find malaria parasites, on the samepatient split every published number in this repo uses.## The rules this notebook enforces1. **The split is inherited, not chosen.** `eval/results/wbc_sweep.json` ->   `fit_patients` is 100 patient ids. The held-out 50 are the complement.2. **Validation comes out of the fit patients**, 80/20 by patient. Checkpoints   and confidence thresholds are chosen there.3. **The held-out 50 are scored once**, at the end, at a threshold already   decided. There is a cell for it, right at the bottom, and it is the last   thing you run. `detector_eval.py` refuses to sweep thresholds on them.The number to beat, on those held-out patients: the patch network at itsshipped 0.75 threshold -- **precision 0.790, recall 0.739, F1 0.763, countbias -6.4%** (`docs/results-parasites.md`).Quote the count bias next to the F1, always. The threshold moved from 0.80 to0.75 not because F1 improved (0.762 against 0.763, the same curve) but because0.80 ran the count 14.4% low, and the count is the denominator of everydensity a clinician reads.## RuntimeSet **Runtime > Change runtime type > T4 GPU** before running anything.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csvimport torchprint("torch", torch.__version__, "| cuda", torch.version.cuda, "| available", torch.cuda.is_available())assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."

## 2. Mount Drive for cachingThe raw dataset is 3.9 GB and the export takes ~25 minutes. Colab wipes itsdisk when the session ends, so both get cached to Drive and later sessionsskip straight to training.Skip this cell if you would rather re-download every session.

In [ ]:
from pathlib import PathUSE_DRIVE = True   # set False to work entirely on Colab's ephemeral diskif USE_DRIVE:    from google.colab import drive    drive.mount('/content/drive')    CACHE = Path('/content/drive/MyDrive/autoscope-cache')else:    CACHE = Path('/content/cache')CACHE.mkdir(parents=True, exist_ok=True)print("cache:", CACHE)!df -h /content | tail -1

## 3. Get the codeThe repo is private and its remote is SSH, which Colab cannot use. Clone overHTTPS with a personal access token instead: GitHub > Settings > Developersettings > Tokens (classic), scope `repo`.The token is read with `getpass`, so it is never written into the notebook andnever saved in the output. **Push the branch first** if you have not already:`git push -u origin b/parasite-detectors`.

In [ ]:
import getpass, osfrom pathlib import PathBRANCH = 'b/parasite-detectors'OWNER_REPO = 'nguefackuriel/autoscope-opencv'if not Path('/content/autoscope-opencv').exists():    token = getpass.getpass('GitHub token (input hidden): ')    !git clone --branch {BRANCH} --single-branch https://{token}@github.com/{OWNER_REPO}.git /content/autoscope-opencv    del token%cd /content/autoscope-opencv!git log --oneline -3

## 4. DependenciesColab already ships torch and torchvision built against its own CUDA. Forcingthe pins in `requirements-detect.txt` would re-download ~3 GB and can break theCUDA build, so we use what Colab has and **record the actual versions** below,so any result stays attributable to a real environment.This is a deliberate divergence from the pinned file. On the AWS instance, usethe pins.

In [ ]:
!pip install -q ultralytics==8.4.142 clearml==2.1.12import json, platform, subprocess, sysimport torch, torchvision, cv2, numpy, ultralyticsENV = {    "python": platform.python_version(),    "torch": torch.__version__,    "torchvision": torchvision.__version__,    "opencv": cv2.__version__,    "numpy": numpy.__version__,    "ultralytics": ultralytics.__version__,    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,}print(json.dumps(ENV, indent=2))Path('/content/autoscope-opencv/eval/results').mkdir(parents=True, exist_ok=True)Path('/content/autoscope-opencv/eval/results/colab_env.json').write_text(json.dumps(ENV, indent=2))

## 5. The datasetThe raw set is needed **every session**, even when the tiles are cached:training uses tiles, but val and holdout scoring run on **whole fields**, so`detector_eval.py` reads the original images.Optionally cache the 3.9 GB archive on Drive (`CACHE_ARCHIVE = True`) to turn aslow internet download into a fast Drive copy. It costs 3.9 GB of a 15 GB freeDrive, so it is off by default.

In [ ]:
from pathlib import Pathimport shutilCACHE_ARCHIVE = False          # True to keep the 3.9 GB zip on DriveARCHIVE = Path('/content/autoscope-opencv/data/raw/ThickBloodSmears_150.zip')CACHED_ARCHIVE = CACHE / 'ThickBloodSmears_150.zip'if CACHE_ARCHIVE and CACHED_ARCHIVE.exists():    print("copying the archive from Drive")    ARCHIVE.parent.mkdir(parents=True, exist_ok=True)    shutil.copy(CACHED_ARCHIVE, ARCHIVE)!python data/download.pyif CACHE_ARCHIVE and ARCHIVE.exists() and not CACHED_ARCHIVE.exists():    print("caching the archive to Drive for next session")    shutil.copy(ARCHIVE, CACHED_ARCHIVE)

### Confirm where the patients actually landed`download.py` extracts to `data/raw/ThickBloodSmears_150/`, while the oldereval scripts scan `data/raw/` directly. `splits.patient_root()` resolves eitherlayout; this cell shows which one you got, and fails loudly rather thansilently reporting a split of zero patients.

In [ ]:
import syssys.path.insert(0, '/content/autoscope-opencv/eval')sys.path.insert(0, '/content/autoscope-opencv/src')import splitsroot = splits.patient_root()allp = splits.all_patients()fit = splits.fit_patients()hold = splits.holdout_patients()train_ids, val_ids = splits.train_val()print("patient root :", root)print("patients     :", len(allp))print("fit / holdout:", len(fit & set(allp)), "/", len(hold))print("train / val  :", len(train_ids), "/", len(val_ids))assert len(allp) == 150, f"expected 150 patients, found {len(allp)}"assert len(hold) == 50, f"expected 50 held out, found {len(hold)}"assert not (set(train_ids) | set(val_ids)) & set(hold), "held-out patient leaked into training"print("\nsplit checks passed")

## 6. Export the tiled dataset -- **this is the decision gate**Cuts 640 px tiles at native resolution from the **fit patients only** andderives a box for every annotated parasite from the stain.**Read the drop rate before you train anything.** It is the fraction ofannotated parasites for which `boxes.py` found no stain peak and thereforeproduced no box. A high number means the detectors would train only oncleanly-stained parasites, and would look better in training than they are.Rough guide: under 10% is fine, 10-25% is worth a look at which parasites arebeing dropped, and above ~35% says the box route is fighting the data and apoint-native (heatmap) detector is the better bet.

In [ ]:
from pathlib import PathTILES = Path('/content/autoscope-opencv/data/derived/detect')CACHED_TILES = CACHE / 'detect_tiles.tar'if not TILES.exists() and CACHED_TILES.exists():    print("restoring exported tiles from Drive, skipping the ~25 min export")    TILES.parent.mkdir(parents=True, exist_ok=True)    !tar -xf {CACHED_TILES} -C /content/autoscope-opencv/data/derivedif not TILES.exists():    !python eval/export_detection.py --images-per-patient 3else:    print("tiles present, skipping export")import jsonm = json.loads((TILES / 'manifest.json').read_text())print(f"\nsplit source           {m['split_source']}")print(f"train / val patients   {m['train_patients']} / {m['val_patients']}")print(f"held out, untouched    {m['holdout_patients_untouched']}")print(f"tiles                  train {m['counts'].get('tiles_train',0)}, val {m['counts'].get('tiles_val',0)}")print(f"boxes                  train {m['counts'].get('boxes_train',0)}, val {m['counts'].get('boxes_val',0)}")print(f"\nannotated parasites    {m['annotated_parasites']}")print(f"boxes derived          {m['boxes_derived']}")print(f"DROP RATE              {m['box_drop_rate']*100:.1f}%   <-- read this")print(f"box side px            p10 {m['box_side_p10_px']}, median {m['box_side_median_px']}, p90 {m['box_side_p90_px']}")

### Cache the tiles to DriveSo the next session starts here instead of at the 3.9 GB download.

In [ ]:
if not CACHED_TILES.exists():    !tar -cf {CACHED_TILES} -C /content/autoscope-opencv/data/derived detect    print("cached to", CACHED_TILES)!du -sh {CACHED_TILES}

### Sanity-check the boxes by eyeNumbers can look fine while the boxes sit next to the parasites rather than onthem. One glance is worth more than the manifest here.

In [ ]:
import cv2 as cv, matplotlib.pyplot as plt, randomfrom pathlib import Pathtile_dir = TILES / 'images' / 'train'label_dir = TILES / 'labels' / 'train'names = [p for p in sorted(tile_dir.glob('*.jpg'))         if (label_dir / (p.stem + '.txt')).read_text().strip()]print(len(names), "tiles carry at least one box")fig, axes = plt.subplots(1, 3, figsize=(16, 6))for ax, path in zip(axes, random.Random(0).sample(names, min(3, len(names)))):    img = cv.cvtColor(cv.imread(str(path)), cv.COLOR_BGR2RGB)    h, w = img.shape[:2]    for line in (label_dir / (path.stem + '.txt')).read_text().splitlines():        _, cx, cy, bw, bh = (float(v) for v in line.split())        x0, y0 = int((cx - bw/2) * w), int((cy - bh/2) * h)        cv.rectangle(img, (x0, y0), (x0 + int(bw*w), y0 + int(bh*h)), (255, 0, 0), 2)    ax.imshow(img); ax.set_title(path.name[:28], fontsize=8); ax.axis('off')plt.tight_layout(); plt.show()

## 7. Train YOLO26Roughly 2-4 hours for 100 epochs on a T4, which fits inside one Colab session.Augmentation is turned down on purpose -- see the docstring in`train/yolo26.py`: mosaic would halve the apparent size of a 10 px parasite.ClearML is off here to keep the notebook frictionless. Drop `--no-clearml` andrun `clearml-init` first if you want the run tracked.

In [ ]:
!python train/yolo26.py --epochs 100 --batch 16 --model yolo26s.pt --no-clearml

In [ ]:
# Keep the weights: Colab will delete them with the session.WEIGHTS = Path('/content/autoscope-opencv/models/yolo26/yolo26/weights/best.pt')assert WEIGHTS.exists(), f"no weights at {WEIGHTS} -- check the training output above"(CACHE / 'weights').mkdir(exist_ok=True)!cp {WEIGHTS} {CACHE}/weights/yolo26_best.ptprint("saved to Drive:", CACHE / 'weights' / 'yolo26_best.pt')

## 8. Choose the threshold -- on VAL, never on holdoutSweeps confidence on the fit patients held back from training. Pick thethreshold you want from this table. `detector_eval.py` will refuse to run thissweep against the held-out split.

In [ ]:
!python eval/detector_eval.py --backend yolo --weights {CACHE}/weights/yolo26_best.pt \    --split val --sweep --per-patient 3

## 9. Faster R-CNN, same tilesSlower: budget ~20-30 min per epoch on a T4. Checkpoints are written everyepoch, so if the session dies you can restart from the last one rather thanfrom scratch. Consider `--epochs 12` for a first pass.

In [ ]:
!python train/frcnn.py --epochs 12 --batch 4 --workers 2 --no-clearml

In [ ]:
import shutilckpts = sorted(Path('/content/autoscope-opencv/models/frcnn').glob('epoch*.pt'))print(len(ckpts), "checkpoints")if ckpts:    shutil.copy(ckpts[-1], CACHE / 'weights' / 'frcnn_last.pt')    print("saved to Drive:", CACHE / 'weights' / 'frcnn_last.pt')

In [ ]:
!python eval/detector_eval.py --backend frcnn --weights {CACHE}/weights/frcnn_last.pt \    --split val --sweep --per-patient 3

## 10. The held-out 50 -- run this **once**Everything above happened on fit patients. This is the only cell that touchesthe held-out 50, and it is the number that goes in the report.Set `THRESHOLD` from the val sweep above **before** running it. Running thisrepeatedly with different thresholds until one looks good is fitting on theheld-out set, and it would invalidate the generalisation claim the wholeproject rests on.

In [ ]:
THRESHOLD = 0.25   # <-- set from the val sweep, then run this cell once!python eval/detector_eval.py --backend yolo --weights {CACHE}/weights/yolo26_best.pt \    --split holdout --threshold {THRESHOLD} --per-patient 3 --tag yolo26_holdout

In [ ]:
THRESHOLD_FRCNN = 0.25   # <-- set from the frcnn val sweep!python eval/detector_eval.py --backend frcnn --weights {CACHE}/weights/frcnn_last.pt \    --split holdout --threshold {THRESHOLD_FRCNN} --per-patient 3 --tag frcnn_holdout

## 11. Collect the resultsCopies the eval json out to Drive so you can commit them and write up thecomparison against the patch network's 0.763.

In [ ]:
import json, shutilfrom pathlib import PathRESULTS = Path('/content/autoscope-opencv/eval/results')(CACHE / 'results').mkdir(exist_ok=True)for f in RESULTS.glob('detector_eval_*.json'):    shutil.copy(f, CACHE / 'results' / f.name)    print("saved", f.name)print(f"\n{'model':<10}{'thr':>6}{'prec':>8}{'recall':>8}{'F1':>8}{'bias%':>9}")for f in sorted(RESULTS.glob('detector_eval_*holdout*.json')):    d = json.loads(f.read_text())    for thr, r in d['by_threshold'].items():        print(f"{d['backend']:<10}{thr:>6}{r['precision']:>8.3f}"              f"{r['recall']:>8.3f}{r['f1']:>8.3f}{r['count_bias_pct']:>+9.1f}")print(f"{'patch CNN':<10}{'0.75':>6}{0.790:>8.3f}{0.739:>8.3f}{0.763:>8.3f}{-6.4:>+9.1f}   <- to beat")